### **Setting Up Workspace**

In [1]:
# src/train_model.py
import pandas as pd
import xgboost as xgb
from sklearn.metrics import classification_report, mean_absolute_error, r2_score
import joblib # For saving our trained models

### **Load Clean Feature Data**

Our first step is to load the hyp_a_features.parquet file that your previous script created.

In [2]:
# --- Step 1: Load Feature Data ---
print("Step 1: Loading features...")

try:
    year = 2018
    df = pd.read_parquet(f'../data/processed/hyp_a_features_advanced.parquet')
except FileNotFoundError:
    print("Error: The feature file was not found.")
    print("Please run the 'feature_engineering.py' script first.")
    exit()

print("Features loaded successfully.")
df.info()

Step 1: Loading features...
Features loaded successfully.
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1994 entries, 2018-01-04 to 2025-09-24
Data columns (total 32 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   day_of_week          1994 non-null   int64  
 1   asia_return          1994 non-null   float64
 2   asia_range           1994 non-null   float64
 3   london_direction     1994 non-null   int64  
 4   london_return        1994 non-null   float64
 5   RSI_14               1994 non-null   float64
 6   MOM_10               1994 non-null   float64
 7   STOCHk_14_3_3        1994 non-null   float64
 8   STOCHd_14_3_3        1994 non-null   float64
 9   STOCHh_14_3_3        1994 non-null   float64
 10  CCI_14_0.015         1994 non-null   float64
 11  ROC_10               1994 non-null   float64
 12  CMO_14               1994 non-null   float64
 13  STOCHRSIk_14_14_3_3  1994 non-null   float64
 14  STOCHRSId_14

In [9]:
df

,day_of_week,asia_return,asia_range,london_direction,london_return,RSI_14,MOM_10,STOCHk_14_3_3,STOCHd_14_3_3,STOCHh_14_3_3,...,WMA_10,DEMA_10,BOP,ATRr_14,PSAR,MACD_12_26_9,MACDh_12_26_9,MACDs_12_26_9,TRIX_30_9,TRIXs_30_9
date,,,,,,,,,,,,,,,,,,,,,
2018-01-04,3,-0.002217,10.84,1,0.003565,45.078922,-6.11,22.171226,15.319033,6.852193,...,1308.874545,1307.859555,0.779851,2.596249,1315.105707,-1.658247,-0.679946,-0.978301,-0.001497,-0.000003
2018-01-05,4,-0.003126,6.02,0,-0.003038,55.991011,1.03,62.873276,68.619618,-5.746342,...,1321.418000,1322.031673,-0.853535,2.453258,1325.527730,2.172012,-0.035823,2.207835,0.005415,0.002398
2018-01-08,0,-0.002128,4.77,1,0.000759,45.015562,-2.39,59.471082,69.017323,-9.546241,...,1319.780000,1319.590210,-0.721854,2.246994,1314.205989,0.393206,-0.220134,0.613340,0.008113,0.008157
2018-01-09,1,0.000212,4.95,0,-0.007216,51.761549,0.41,66.047107,63.291493,2.755614,...,1318.772182,1318.833379,0.402878,1.891777,1315.422784,-0.006538,0.033848,-0.040386,0.003413,0.004123
2018-01-10,2,-0.003075,6.25,1,0.006139,36.431761,-3.03,28.646518,36.936254,-8.289736,...,1310.837818,1310.009542,-0.506173,2.116085,1312.483658,-1.662555,-0.042516,-1.620039,-0.007136,-0.005239
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-18,3,0.000822,25.40,1,0.001320,39.790183,-32.85,20.906170,23.365826,-2.459656,...,3660.438909,3657.244223,-0.821918,11.136647,3698.285205,-5.523691,-1.236808,-4.286883,0.003886,0.007489
2025-09-19,4,0.004817,27.93,0,-0.001251,55.276912,15.26,70.289427,50.194429,20.094998,...,3647.289091,3649.139056,0.809221,9.654557,3628.349400,-3.597413,2.084138,-5.681551,-0.012482,-0.010363
2025-09-22,0,0.001815,13.78,1,0.007079,66.165683,13.53,87.811628,87.514731,0.296897,...,3689.681818,3694.737658,-0.561102,8.396011,3697.470000,9.897547,1.518816,8.378730,0.001582,-0.003402


### **Step 2: Define Features (X) and Targets (y) and Split the Data**

This is the most important conceptual step in machine learning. We need to separate our data into two groups:

- `X` (The Features): The information the model will use to make a prediction (e.g., asia_return, rsi_at_asia_close).
- `y` (The Target): The answer the model is trying to predict (e.g., london_direction).

**Crucially, for time-series data, we CANNOT split the data randomly**. We must split it chronologically to simulate reality. We train on the past and test on the more recent "future".

In [3]:
# --- Step 2: Define Features, Targets, and Split Data ---
print("\nStep 2: Preparing data for training...")

# 'X' is our feature set. We drop the two target columns.
X = df.drop(columns=['london_direction', 'london_return'])

# We have two separate targets we want to predict.
y_class = df['london_direction']  # For our classification model
y_reg = df['london_return']      # For our regression model

# --- The Time-Series Split ---
# We will use the first 80% of the data for training and the last 20% for testing.
# This ensures we are always testing on data that comes after our training data.
train_size = int(len(df) * 0.8)

X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train_class, y_test_class = y_class.iloc[:train_size], y_class.iloc[train_size:]
y_train_reg, y_test_reg = y_reg.iloc[:train_size], y_reg.iloc[train_size:]

print(f"Data split into training and testing sets:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")


Step 2: Preparing data for training...
Data split into training and testing sets:
X_train shape: (1595, 30)
X_test shape:  (399, 30)


### **Step 3: Train the Classification Model (Hypothesis A1)**

Now we'll teach our first model to predict the direction (1 or 0). We will use XGBClassifier.

In [6]:
# --- Step 3: Train Classification Model ---
print("\n--- Training Classification Model (Hypothesis A1) ---")

# Initialize the XGBoost Classifier model with some standard parameters.
# 'objective' tells it to perform binary (two-class) classification.
# 'eval_metric' is the metric used to stop training early if it's not improving.
model_class = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_estimators=1000, # Number of decision trees to build.
    learning_rate=0.001,
    max_depth=3,
    use_label_encoder=False,
    random_state=42
)

# Train the model on our training data.
model_class.fit(X_train, y_train_class)

# --- Evaluate the Classification Model ---
print("\n--- Evaluating Classification Model ---")

# Make predictions on the unseen test data.
y_pred_class = model_class.predict(X_test)

# Print a report showing key metrics.
# Precision: Of all the "bullish" predictions, how many were correct?
# Recall: Of all the actual bullish days, how many did we correctly identify?
# F1-Score: A combined score of precision and recall.
print(classification_report(y_test_class, y_pred_class, target_names=['Bearish (0)', 'Bullish (1)']))


--- Training Classification Model (Hypothesis A1) ---


c:\Users\mecha\anaconda3\envs\trade_env\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:03:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- Evaluating Classification Model ---
              precision    recall  f1-score   support

 Bearish (0)       0.51      0.44      0.47       184
 Bullish (1)       0.57      0.64      0.60       215

    accuracy                           0.55       399
   macro avg       0.54      0.54      0.54       399
weighted avg       0.54      0.55      0.54       399



### **Step 4: Train the Regression Model (Hypothesis A2)**

Next, we'll teach our second model to predict the actual return value. We will use XGBRegressor.

In [ ]:
# --- Step 4: Train Regression Model ---
print("\n--- Training Regression Model (Hypothesis A2) ---")

# Initialize the XGBoost Regressor model.
# 'objective' tells it to minimize the squared error, which is standard for regression.
model_reg = xgb.XGBRegressor(
    objective='reg:squarederror',
    eval_metric='rmse', # Root Mean Squared Error
    n_estimators=1000,
    learning_rate=0.00001,
    max_depth=5,
    random_state=42
)

# Train the model on our training data.
model_reg.fit(X_train, y_train_reg)

# --- Evaluate the Regression Model ---
print("\n--- Evaluating Regression Model ---")

# Make predictions on the unseen test data.
y_pred_reg = model_reg.predict(X_test)

# Calculate and print key metrics.
mae = mean_absolute_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print(f"Mean Absolute Error (MAE): {mae:.6f}")
print("MAE tells us, on average, how far off our return prediction was in percentage points.")
print(f"R-squared (R2 Score): {r2:.4f}")
print("R2 Score tells us how much of the variance in the returns our model can explain (closer to 1 is better).")


--- Training Regression Model (Hypothesis A2) ---

--- Evaluating Regression Model ---
Mean Absolute Error (MAE): 0.004786
MAE tells us, on average, how far off our return prediction was in percentage points.
R-squared (R2 Score): -0.0534
R2 Score tells us how much of the variance in the returns our model can explain (closer to 1 is better).


--- Training Regression Model (Hypothesis A2) ---

--- Evaluating Regression Model ---
Mean Absolute Error (MAE): 0.005979
MAE tells us, on average, how far off our return prediction was in percentage points.
R-squared (R2 Score): -0.7396
R2 Score tells us how much of the variance in the returns our model can explain (closer to 1 is better).

**Step 5: Save Your Trained Models**

The final step is to save our two trained models so we can load them later in our backtesting script without having to retrain them.

In [7]:
# --- Step 5: Save Trained Models ---
print("\nStep 5: Saving models...")

# Define the paths where the models will be saved.
class_model_path = f'../models/xgb_classifier_hyp_a_{year}_present_TUNED_NEW.joblib'
# reg_model_path = f'../models/xgb_regressor_hyp_a_{year}_present.joblib'

# Use joblib to dump the trained model objects into files.
joblib.dump(model_class, class_model_path)
# joblib.dump(model_reg, reg_model_path)

print(f"Classification model saved to: {class_model_path}")
# print(f"Regression model saved to: {reg_model_path}")


Step 5: Saving models...
Classification model saved to: ../models/xgb_classifier_hyp_a_2018_present_TUNED_NEW.joblib
